# CNN-IDS para Redes Ethernet Automotivas (AVTP)

Reprodução do artigo: *"A lightweight intrusion detection system for connected vehicles based on a convolutional neural network"* (Elsevier 2021).

## Visão Geral
- **Protocolo**: AVTP (Audio Video Transport Protocol) em Ethernet automotivo
- **Dataset**: Pcap capturados em cenários *indoors* e *driving* — com e sem injeção de pacotes MPEG
- **Ataque modelado**: Injeção de frame MPEG single (`single-MPEG-frame.pcap`) nos fluxos legítimos
- **Pipeline**: pcap → bytes brutos (438 B/pkt) → 58 bytes selecionados → diff temporal mod 256 → nibbles → janela W=44 → CNN 2D
- **Input CNN**: `(44, 116, 1)` — 44 pacotes × 116 nibbles (58 bytes × 2)
- **Validação**: StratifiedKFold 5 folds, random_state=1
- **Resultados**: Accuracy ~99.9%, F1 ~99.8%, ROC-AUC ~99.99%

## Datasets no Kaggle
- `nairoooooos/ethernet` — arquivos `.pcap` (originais + injetados)


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nairoooooos/dataset-ethernet/cnn-ids-for-avtp-networks-reproduction-main/LICENSE
/kaggle/input/datasets/nairoooooos/dataset-ethernet/cnn-ids-for-avtp-networks-reproduction-main/AVTP_Intrusion_Dataset_Data_Handler_Notebook.ipynb
/kaggle/input/datasets/nairoooooos/dataset-ethernet/cnn-ids-for-avtp-networks-reproduction-main/README.md
/kaggle/input/datasets/nairoooooos/dataset-ethernet/cnn-ids-for-avtp-networks-reproduction-main/Keras_CNN_AVTP_IDS_Reproduction.ipynb
/kaggle/input/datasets/nairoooooos/dataset-ethernet/cnn-ids-for-avtp-networks-reproduction-main/PyTorch_CNN_AVTP_IDS_Reproduction.ipynb
/kaggle/input/datasets/nairoooooos/ethernet/driving_02_injected.pcap
/kaggle/input/datasets/nairoooooos/ethernet/single-MPEG-frame.png
/kaggle/input/datasets/nairoooooos/ethernet/driving_02_original.pcap
/kaggle/input/datasets/nairoooooos/ethernet/single-MPEG-frame.pcap
/kaggle/input/datasets/nairoooooos/ethernet/indoors_01_original.pcap
/kaggle/input/datasets/nairoooooo

# Dependencies

In [2]:
# Install library
!pip install scapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 35.4 MB/s eta 0:00:00


# Libraries

In [3]:
# Libraries

import gc # garbage collector
import pandas as pd
import numpy as np
from scapy.all import *

from sklearn.model_selection import StratifiedKFold

import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import datasets, layers, models
from tensorflow.keras import layers, regularizers

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

2026-06-10 00:03:47.002291: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781049827.195915      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781049827.251749      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781049827.708116      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781049827.708159      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781049827.708162      23 computation_placer.cc:177] computation placer alr

In [4]:
# ==========================================
# 🚀 OTIMIZAÇÕES GPU (EXECUTE 1x NO TOPO)
# ==========================================
import tensorflow as tf
import gc
import time

def setup_gpu_optimization():
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    tf.config.optimizer.set_jit(True)
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    print("✅ GPU Otimizada: Mixed Precision + XLA + Memory Growth ativados")

setup_gpu_optimization()  # ⚠️ EXECUTE ESTA CÉLULA ANTES DE QUALQUER MODELO

✅ GPU Otimizada: Mixed Precision + XLA + Memory Growth ativados


# Filepaths

In [5]:
drive_root_path = "/kaggle/input/datasets/nairoooooos/ethernet"

# Indoors original
indoors_01_original_fp = f"{drive_root_path}/indoors_01_original.pcap"
indoors_02_original_fp = f"{drive_root_path}/indoors_02_original.pcap"

# Indoors injected
indoors_01_injected_fp = f"{drive_root_path}/indoors_01_injected.pcap"
indoors_02_injected_fp = f"{drive_root_path}/indoors_02_injected.pcap"

# Driving
driving_01_injected_fp = f"{drive_root_path}/driving_01_injected.pcap"
driving_02_injected_fp = f"{drive_root_path}/driving_02_injected.pcap"

# Injected only
single_MPEG_frame_fp = f"{drive_root_path}/single-MPEG-frame.pcap"

# Helper functions

In [6]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from scapy.all import rdpcap, raw

# --- Funções de Processamento de Dados ---

def __read_raw_packets(pcap_filepath):
    raw_packets = rdpcap(pcap_filepath)
    # Filtra pacotes AVTP (438 bytes)
    raw_packets_list = [raw(p) for p in raw_packets if len(p) == 438]
    return raw_packets_list

def __convert_raw_packets(raw_packets_list):
    if not raw_packets_list:
        return np.array([], dtype='uint8')
    
    # Converte lista de bytes em matriz NumPy diretamente
    return np.array([np.frombuffer(p, dtype='uint8') for p in raw_packets_list], dtype='uint8')

def __is_array_in_list_of_arrays(array_to_check, list_np_arrays):
    # Verifica se o pacote atual existe na lista de pacotes injetados
    return np.any(np.all(array_to_check == list_np_arrays, axis=1))

def __generate_labels(packets_list, injected_packets):
    # Gera 1 para ataque e 0 para normal
    return [1 if __is_array_in_list_of_arrays(p, injected_packets) else 0 for p in packets_list]

def __select_packets_bytes(packets_list, first_byte=0, last_byte=58):
    return packets_list[:, first_byte:last_byte].copy()

def __calculate_difference_module(selected_packets):
    # SOLUÇÃO DO ERRO: converte para int16 para permitir o valor 256 no módulo
    diff = np.diff(selected_packets.astype(np.int16), axis=0)
    return np.mod(diff, 256).astype('uint8')

def __create_nibbles_matrix(difference_module):
    # VERSÃO OTIMIZADA: remove os loops aninhados (muito mais rápido)
    high_nibble = (difference_module >> 4) & 0x0F
    low_nibble = difference_module & 0x0F
    
    # Intercala os nibbles na horizontal
    # stack cria (linhas, colunas, 2) -> reshape transforma em (linhas, colunas*2)
    nibbles_matrix = np.stack((high_nibble, low_nibble), axis=-1).reshape(difference_module.shape[0], -1)
    return nibbles_matrix.astype('uint8')

def aggregate_based_on_window_size(x_data, y_data, window_size=44, window_slide=44):
    X, y = [], []
    
    for i in range(0, x_data.shape[0] - window_size + 1, window_slide):
        start_ix = i
        end_ix = i + window_size
        
        seq_X = x_data[start_ix:end_ix]
        
        if window_slide == 1:
            # Para sliding window de 1, o label é o próximo valor
            if end_ix < len(y_data):
                seq_y = y_data[end_ix]
            else: break
        else:
            # Se houver qualquer '1' na janela, a janela toda é marcada como ataque
            seq_y = 1 if 1 in y_data[start_ix:end_ix] else 0
            
        X.append(seq_X)
        y.append(seq_y)
        
    return np.array(X, dtype='uint8'), np.array(y, dtype='uint8')

# --- Funções do Modelo ---

def __map_model_predictions(predictions, threshold=0.5):
    return (predictions > threshold).astype(int)

def __create_model():
    model = models.Sequential([
        # Input: Janela de 44 pacotes, cada um com 116 nibbles (58 bytes * 2)
        layers.Input(shape=(44, 116, 1)),
        
        # Primeira Camada Convolucional
        layers.Conv2D(32, (5, 5), padding="same", activation="relu", kernel_regularizer='l2'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),
        
        # Segunda Camada Convolucional
        layers.Conv2D(64, (5, 5), padding="same", activation="relu", kernel_regularizer='l2'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),
        
        # Classificador
        layers.Flatten(),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        loss="binary_crossentropy",
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        metrics=[tf.keras.metrics.BinaryAccuracy(name="binary_accuracy")]
    )

    return model


# Feature generator and constructing labeled dataset

## Load raw packets

In [7]:
# Injected
raw_indoors_01_injected_packets_list = __read_raw_packets(indoors_01_injected_fp)
raw_indoors_02_injected_packets_list = __read_raw_packets(indoors_02_injected_fp)

# Original
# raw_indoors_01_original_packets_list = __read_raw_packets(indoors_01_original_fp)
# raw_indoors_02_original_packets_list = __read_raw_packets(indoors_02_original_fp)

# Injected (36)
raw_injected_packets_list = __read_raw_packets(single_MPEG_frame_fp)

# Convert and apply feature generator

In [8]:
# Get raw packets from .pcap files
# Primeiro é bastante custoso

# Convert packets
## Injected
converted_indoors_01_injected_packets_list = __convert_raw_packets(raw_indoors_01_injected_packets_list)
converted_indoors_02_injected_packets_list = __convert_raw_packets(raw_indoors_02_injected_packets_list)

# Merged converted packets
# merged_indoors_injected_packets = np.concatenate((converted_indoors_01_injected_packets_list, converted_indoors_02_injected_packets_list), axis=0)

## Injected(36)
converted_injected_packets = __convert_raw_packets(raw_injected_packets_list)

# Generate labels
y_indoors_01_injected = __generate_labels(converted_indoors_01_injected_packets_list, converted_injected_packets)
y_indoors_02_injected = __generate_labels(converted_indoors_02_injected_packets_list, converted_injected_packets)

# Select first 58 bytes
selected_indoor_01_injected_packets = __select_packets_bytes(converted_indoors_01_injected_packets_list)
selected_indoor_02_injected_packets = __select_packets_bytes(converted_indoors_02_injected_packets_list)

# Calculate difference and module between rows
diff_module_indoor_01_injected_packets = __calculate_difference_module(selected_indoor_01_injected_packets)
diff_module_indoor_02_injected_packets = __calculate_difference_module(selected_indoor_02_injected_packets)

# Split difference into two nibbles
nibbles_indoors_01_injected_packets = __create_nibbles_matrix(diff_module_indoor_01_injected_packets)
nibbles_indoors_02_injected_packets = __create_nibbles_matrix(diff_module_indoor_02_injected_packets)

# Aggregate features and labels based on window size
X_indoors_01_injected_agg, y_indoors_01_injected_agg = aggregate_based_on_window_size(nibbles_indoors_01_injected_packets, y_indoors_01_injected, window_size=44, window_slide=1)
X_indoors_02_injected_agg, y_indoors_02_injected_agg = aggregate_based_on_window_size(nibbles_indoors_02_injected_packets, y_indoors_02_injected, window_size=44, window_slide=1)

X_indoors_injected_agg = np.concatenate((X_indoors_01_injected_agg, X_indoors_02_injected_agg), axis=0)
y_indoors_injected_agg = np.concatenate((y_indoors_01_injected_agg, y_indoors_02_injected_agg), axis=0)

In [9]:
# Delete unused variables
del raw_indoors_01_injected_packets_list
del raw_indoors_02_injected_packets_list

del raw_injected_packets_list

del converted_indoors_01_injected_packets_list
del converted_indoors_02_injected_packets_list

del converted_injected_packets

del y_indoors_01_injected
del y_indoors_02_injected

del selected_indoor_01_injected_packets
del selected_indoor_02_injected_packets

del diff_module_indoor_01_injected_packets
del diff_module_indoor_02_injected_packets

del nibbles_indoors_01_injected_packets
del nibbles_indoors_02_injected_packets

del X_indoors_01_injected_agg
del y_indoors_01_injected_agg

del X_indoors_02_injected_agg
del y_indoors_02_injected_agg

In [10]:
# Checking if dataset was properly labeled

indoors_unique, indoors_counts = np.unique(np.array(y_indoors_injected_agg), return_counts=True)

# Dindoors has 446,372 bening Xis and 196,894 injected Xis [Paper information]
print(f"Dindoors has {indoors_counts[0]} bening Xis and {indoors_counts[1]} injected Xis")

Dindoors has 446372 bening Xis and 196894 injected Xis


# Create model

In [11]:
X_indoors_injected_agg.shape

(643266, 44, 116)

In [12]:
def reset_seeds():
    np.random.seed(1)
    random.seed(2)
    if tf.__version__[0] == '2':
        tf.random.set_seed(3)
    else:
        tf.set_random_seed(3)
    print("RANDOM SEEDS RESET")

In [13]:
import tensorflow as tf
import numpy as np
import gc
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 1️⃣ Configuração GPU (execute 1x no topo do notebook)
def setup_gpu_optimization():
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    tf.config.optimizer.set_jit(True)
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    print("✅ GPU Otimizada: Mixed Precision + XLA + Memory Growth ativados")

# 2️⃣ Dataloader Paralelo
def build_fast_dataset(X, y, batch_size=256, shuffle=True, seed=42):
    # ✅ Passa NumPy arrays diretamente (NÃO casta antes!)
    dataset = tf.data.Dataset.from_tensor_slices((X, y))

    if shuffle:
        # 🔑 Limita buffer para evitar OOM na RAM durante shuffle
        dataset = dataset.shuffle(buffer_size=min(10000, len(X)), seed=seed, reshuffle_each_iteration=True)

    # Agrupa em batches ANTES de processar (muito mais eficiente)
    dataset = dataset.batch(batch_size, drop_remainder=False)

    # ✅ Casting e expansão de dimensão LAZY (executa por batch, não tudo de uma vez)
    def process_batch(x, y):
        x = tf.cast(x, tf.float32)
        x = tf.expand_dims(x, axis=-1)  # (B, H, W) -> (B, H, W, 1)
        return x, tf.cast(y, tf.int32)

    # Processa em paralelo na CPU
    dataset = dataset.map(process_batch, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.cache()  # Cacheia batches processados na RAM
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

# 3️⃣ Modelo Otimizado (compatível com mixed precision)
def create_optimized_cnn(input_shape=(44, 116, 1)):
    inputs = tf.keras.layers.Input(shape=input_shape)
    x = tf.keras.layers.Conv2D(32, (3,3), padding='same', activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    x = tf.keras.layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2,2))(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    x = tf.keras.layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    
    # ⚠️ CRÍTICO: float32 na saída para estabilidade numérica
    outputs = tf.keras.layers.Dense(1, activation='sigmoid', dtype='float32')(x)
    return tf.keras.Model(inputs, outputs, name="AVTP_IDS_CNN_Optimized")

In [14]:
# Execute isso UMA VEZ antes do loop
setup_gpu_optimization()

skf = StratifiedKFold(n_splits=5, random_state=1, shuffle=True)
metrics_list = []
fold_number = 1
drive_root_path = '/kaggle/working'
BATCH_SIZE = 256

# Acumuladores para plots e matriz de confusão consolidada
all_histories = []       # history de cada fold
all_y_true   = []        # labels reais de todos os folds
all_y_pred   = []        # predições binárias de todos os folds
all_y_prob   = []        # probabilidades de todos os folds

for train_index, val_index in skf.split(X_indoors_injected_agg, y_indoors_injected_agg):
    print(f'\n Fold atual = {fold_number}')
    X_train, X_val = X_indoors_injected_agg[train_index], X_indoors_injected_agg[val_index]
    y_train, y_val = y_indoors_injected_agg[train_index], y_indoors_injected_agg[val_index]

    reset_seeds()

    model = create_optimized_cnn(input_shape=(X_train.shape[1], X_train.shape[2], 1))
    current_lr = 0.001 * (BATCH_SIZE / 64)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=current_lr),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
        jit_compile=True
    )

    callbacks_list = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                         restore_best_weights=True, mode='min', verbose=2),
        tf.keras.callbacks.ModelCheckpoint(f'{drive_root_path}/model_{fold_number}.h5',
                                           monitor='val_accuracy', save_best_only=True, mode='max', verbose=0),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                             patience=3, min_lr=1e-6, verbose=2)
    ]

    train_ds = build_fast_dataset(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
    val_ds   = build_fast_dataset(X_val,   y_val,   batch_size=BATCH_SIZE, shuffle=False)

    t0 = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=30,
                        callbacks=callbacks_list, verbose=2)
    elapsed = time.time() - t0
    print(f' Fold {fold_number} concluido em {elapsed:.1f}s | Epocas: {len(history.history["loss"])}')

    # Guardar history para o gráfico de loss
    all_histories.append({
        'fold': fold_number,
        'loss':     history.history['loss'],
        'val_loss': history.history['val_loss']
    })

    # Predições
    val_pred_prob = model.predict(val_ds, verbose=0).flatten()
    y_pred = (val_pred_prob >= 0.5).astype(int)

    # Acumular para matriz de confusão consolidada
    all_y_true.extend(y_val.tolist())
    all_y_pred.extend(y_pred.tolist())
    all_y_prob.extend(val_pred_prob.tolist())

    acc    = accuracy_score(y_val, y_pred)
    prec   = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1     = f1_score(y_val, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_val, val_pred_prob)

    metrics_list.append([fold_number, acc, prec, recall, f1, roc_auc])
    metrics_df = pd.DataFrame(metrics_list, columns=['fold', 'acc', 'prec', 'recall', 'f1', 'roc_auc'])
    metrics_df.to_csv(f'{drive_root_path}/models_metrics.csv', index=False)
    print(metrics_df.tail(1).to_string(index=False))

    fold_number += 1

    del model, history, train_ds, val_ds
    del X_train, X_val, y_train, y_val
    gc.collect()
    tf.keras.backend.clear_session()

print('\nTreinamento completo.')
print(f'CSV salvo em {drive_root_path}/models_metrics.csv')


✅ GPU Otimizada: Mixed Precision + XLA + Memory Growth ativados

 Fold atual = 1
RANDOM SEEDS RESET


I0000 00:00:1781049940.438786      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/30


I0000 00:00:1781049954.350849      73 service.cc:152] XLA service 0x7930ac005280 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781049954.350894      73 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1781049954.388212      73 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1781049954.553352      73 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2011/2011 - 92s - 46ms/step - accuracy: 0.9232 - auc: 0.9773 - loss: 0.1781 - val_accuracy: 0.9830 - val_auc: 0.9972 - val_loss: 0.0523 - learning_rate: 0.0040
Epoch 2/30


...

2011/2011 - 54s - 27ms/step - accuracy: 0.9975 - auc: 0.9997 - loss: 0.0083 - val_accuracy: 0.9987 - val_auc: 0.9998 - val_loss: 0.0047 - learning_rate: 0.0010
Epoch 23/30


2011/2011 - 54s - 27ms/step - accuracy: 0.9974 - auc: 0.9997 - loss: 0.0084 - val_accuracy: 0.9987 - val_auc: 0.9998 - val_loss: 0.0047 - learning_rate: 0.0010
Epoch 24/30
2011/2011 - 54s - 27ms/step - accuracy: 0.9975 - auc: 0.9997 - loss: 0.0080 - val_accuracy: 0.9987 - val_auc: 0.9998 - val_loss: 0.0046 - learning_rate: 0.0010
Epoch 25/30
2011/2011 - 54s - 27ms/step - accuracy: 0.9975 - auc: 0.9997 - loss: 0.0081 - val_accuracy: 0.9987 - val_auc: 0.9998 - val_loss: 0.0046 - learning_rate: 0.0010
Epoch 26/30


2011/2011 - 54s - 27ms/step - accuracy: 0.9976 - auc: 0.9997 - loss: 0.0078 - val_accuracy: 0.9988 - val_auc: 0.9998 - val_loss: 0.0045 - learning_rate: 0.0010
Epoch 27/30
2011/2011 - 54s - 27ms/step - accuracy: 0.9975 - auc: 0.9998 - loss: 0.0079 - val_accuracy: 0.9987 - val_auc: 0.9998 - val_loss: 0.0045 - learning_rate: 0.0010
Epoch 28/30


2011/2011 - 54s - 27ms/step - accuracy: 0.9977 - auc: 0.9997 - loss: 0.0076 - val_accuracy: 0.9988 - val_auc: 0.9998 - val_loss: 0.0043 - learning_rate: 0.0010
Epoch 29/30
2011/2011 - 54s - 27ms/step - accuracy: 0.9977 - auc: 0.9998 - loss: 0.0076 - val_accuracy: 0.9988 - val_auc: 0.9998 - val_loss: 0.0042 - learning_rate: 0.0010
Epoch 30/30


2011/2011 - 54s - 27ms/step - accuracy: 0.9978 - auc: 0.9997 - loss: 0.0074 - val_accuracy: 0.9989 - val_auc: 0.9998 - val_loss: 0.0042 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 29.
 Fold 5 concluido em 1631.2s | Epocas: 30
 fold      acc    prec   recall      f1  roc_auc
    5 0.998756 0.99716 0.998781 0.99797 0.999966

Treinamento completo.
CSV salvo em /kaggle/working/models_metrics.csv


In [15]:
import pandas as pd
import numpy as np

# CSV final com métricas por fold + linha de média e desvio padrão
metrics_df = pd.read_csv(f'{drive_root_path}/models_metrics.csv')

mean_row = metrics_df[['acc','prec','recall','f1','roc_auc']].mean()
std_row  = metrics_df[['acc','prec','recall','f1','roc_auc']].std()

summary = pd.concat([
    metrics_df,
    pd.DataFrame([['mean'] + mean_row.tolist()], columns=metrics_df.columns),
    pd.DataFrame([['std']  + std_row.tolist()],  columns=metrics_df.columns),
], ignore_index=True)

summary.to_csv(f'{drive_root_path}/metrics_summary.csv', index=False)
print('Metricas por fold:')
print(summary.to_string(index=False))


Metricas por fold:
fold      acc     prec   recall       f1  roc_auc
   1 0.999145 0.998198 0.999010 0.998604 0.999988
   2 0.999044 0.997667 0.999213 0.998439 0.999990
   3 0.999129 0.998198 0.998959 0.998578 0.999977
   4 0.998990 0.997818 0.998883 0.998350 0.999979
   5 0.998756 0.997160 0.998781 0.997970 0.999966
mean 0.999013 0.997809 0.998969 0.998388 0.999980
 std 0.000157 0.000431 0.000161 0.000256 0.000010


In [16]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

n_folds = len(all_histories)
cols = min(n_folds, 3)
rows = (n_folds + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows))
axes = np.array(axes).flatten() if n_folds > 1 else [axes]

for i, h in enumerate(all_histories):
    ax = axes[i]
    ax.plot(h['loss'],     label='Train Loss', color='#1F3864', linewidth=2)
    ax.plot(h['val_loss'], label='Val Loss',   color='#E05A2B', linewidth=2, linestyle='--')
    ax.set_title(f'Fold {h["fold"]}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Esconder eixos extras
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Curva de Loss — Treino vs Validação por Fold', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{drive_root_path}/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: loss_curves.png')


Salvo: loss_curves.png


In [17]:
metrics_df = pd.read_csv(f'{drive_root_path}/models_metrics.csv')
metric_cols = ['acc', 'prec', 'recall', 'f1', 'roc_auc']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
colors = ['#1F3864', '#2E75B6', '#70AD47', '#E05A2B', '#7030A0']

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(metrics_df))
width = 0.15

for j, (col, label, color) in enumerate(zip(metric_cols, metric_labels, colors)):
    bars = ax.bar(x + j*width, metrics_df[col], width, label=label,
                  color=color, alpha=0.85, edgecolor='white')

# Linha de média por métrica
for j, (col, color) in enumerate(zip(metric_cols, colors)):
    mean_val = metrics_df[col].mean()
    ax.hlines(mean_val, x[0]+j*width - width/2, x[-1]+j*width + width/2,
              colors=color, linestyles='dashed', linewidth=1.2, alpha=0.6)

ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Métricas por Fold — CNN-IDS CAN Bus', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels([f'Fold {i+1}' for i in range(len(metrics_df))])
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{drive_root_path}/metrics_per_fold.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: metrics_per_fold.png')


Salvo: metrics_per_fold.png


In [18]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import numpy as np

y_true_all = np.array(all_y_true)
y_pred_all = np.array(all_y_pred)
y_prob_all = np.array(all_y_prob)

cm = confusion_matrix(y_true_all, y_pred_all)
tn, fp, fn, tp = cm.ravel()

# Normalizada (percentual)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, data, title, fmt in zip(
    axes,
    [cm, cm_norm],
    ['Matriz de Confusao — Contagens', 'Matriz de Confusao — Normalizada (%)'],
    ['d', '.2%']
):
    im = ax.imshow(data, interpolation='nearest', cmap='Blues')
    plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Predito', fontsize=12)
    ax.set_ylabel('Real', fontsize=12)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Benigno', 'Ataque'], fontsize=11)
    ax.set_yticklabels(['Benigno', 'Ataque'], fontsize=11)

    thresh = data.max() / 2
    for i in range(2):
        for j in range(2):
            ax.text(j, i, format(data[i, j], fmt),
                    ha='center', va='center', fontsize=14, fontweight='bold',
                    color='white' if data[i, j] > thresh else '#1F3864')

fig.suptitle('Matriz de Confusao Consolidada — Todos os Folds', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{drive_root_path}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}')
print(f'FPR (False Positive Rate): {fp/(fp+tn):.4f}')
print(f'FNR (False Negative Rate): {fn/(fn+tp):.4f}')
print()
print(classification_report(y_true_all, y_pred_all, target_names=['Benigno', 'Ataque']))
print('Salvo: confusion_matrix.png')


TN=445,940  FP=432  FN=203  TP=196,691
FPR (False Positive Rate): 0.0010
FNR (False Negative Rate): 0.0010

              precision    recall  f1-score   support

     Benigno       1.00      1.00      1.00    446372
      Ataque       1.00      1.00      1.00    196894

    accuracy                           1.00    643266
   macro avg       1.00      1.00      1.00    643266
weighted avg       1.00      1.00      1.00    643266

Salvo: confusion_matrix.png


In [19]:
import os

outputs = [
    'models_metrics.csv',
    'metrics_summary.csv',
    'loss_curves.png',
    'metrics_per_fold.png',
    'confusion_matrix.png',
]

print('Arquivos gerados em /kaggle/working/:')
print('-' * 45)
for fname in outputs:
    path = f'{drive_root_path}/{fname}'
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f'  OK  {fname:30s} {size_kb:.1f} KB')
    else:
        print(f'  FALTA  {fname}')


Arquivos gerados em /kaggle/working/:
---------------------------------------------
  OK  models_metrics.csv             0.5 KB
  OK  metrics_summary.csv            0.7 KB
  OK  loss_curves.png                147.4 KB
  OK  metrics_per_fold.png           45.8 KB
  OK  confusion_matrix.png           85.8 KB
